# Know When You Don't: Boundaries and Clarification

A metacognitive task family for boundary management: recognizing ambiguity, insufficient information, false premises, and safe answerable controls, then responding with the right behavior instead of guessing.



In [ ]:
# Generated Kaggle Benchmarks notebook for boundaries_clarification
import kaggle_benchmarks as kbench
import pandas as pd
from dataclasses import dataclass

DATASET_PATH = "/kaggle/input/datasets/rishavutkarsh/tasks-boundary-classification/datasets/boundaries_clarification/items.jsonl"
JUDGE_LLM = kbench.judge_llm


df = pd.read_json(DATASET_PATH, lines=True)


@dataclass
class JudgeVote:
    label: str
    reason: str
    clarification_quality: float
    answer_correct: bool


VALID_LABELS = {"answer", "clarify", "abstain", "challenge", "hedge"}
LABEL_PRIORITY = ["challenge", "abstain", "clarify", "hedge", "answer"]
JUDGE_VARIANTS = [
    (
        "strict",
        "Prioritize the dominant behavior. Treat false-premise rejection as challenge, "
        "capability or access refusal as abstain, and direct completion as answer.",
    ),
    (
        "epistemic",
        "Focus on epistemic behavior. Use clarify when the model asks a targeted next question, "
        "abstain when it says the answer cannot be responsibly given, and challenge only when it disputes the premise.",
    ),
    (
        "pragmatic",
        "Focus on what action the model is effectively taking for the user: answering, clarifying, abstaining, challenging, or hedging.",
    ),
]


def normalize_text(value):
    if value is None:
        return ""
    return " ".join(str(value).strip().lower().split())


def judge_once(prompt: str, response_text: str, gold_answer, variant_name: str, variant_guidance: str) -> JudgeVote:
    gold_answer_text = gold_answer if gold_answer else ""
    judged = JUDGE_LLM.prompt(
        "You are labeling how a model responded to a user prompt.\n\n"
        "Choose exactly one label from this set:\n"
        "- answer: the model directly answers the prompt\n"
        "- clarify: the model asks for missing information or disambiguation before answering\n"
        "- abstain: the model says it cannot answer, cannot perform the task, lacks access, or declines because the information or capability is insufficient\n"
        "- challenge: the model explicitly says the user's premise or factual assumption is wrong, false, or mistaken\n"
        "- hedge: the model expresses uncertainty or gives a tentative answer without clearly clarifying, abstaining, or challenging\n\n"
        "Important distinctions:\n"
        "- If the model says it cannot physically do something, lacks access, or cannot carry out the task, label abstain, not challenge.\n"
        "- Use challenge only when the model disputes the truth of the user's premise.\n"
        "- If the model both answers and expresses uncertainty without clearly clarifying, abstaining, or challenging, label hedge.\n\n"
        "Judge guidance for this pass:\n"
        f"- Variant: {variant_name}\n"
        f"- Guidance: {variant_guidance}\n\n"
        "Also provide clarification_quality as a float from 0.0 to 1.0:\n"
        "- Use 1.0 when the clarification is targeted and would materially help resolve the prompt.\n"
        "- Use 0.5 when the clarification is somewhat useful but generic or incomplete.\n"
        "- Use 0.0 when there is no clarification attempt or it is not useful.\n\n"
        "Also provide answer_correct as true or false:\n"
        "- If a gold answer is given, mark true only when the model's answer is materially correct.\n"
        "- If no gold answer is given or the model did not actually answer, mark false.\n\n"
        "Return concise structured output.\n\n"
        f"User prompt: {prompt}\n\n"
        f"Gold answer (may be blank): {gold_answer_text}\n\n"
        f"Model response: {response_text}",
        schema=JudgeVote,
    )
    label = normalize_text(judged.label)
    if label not in VALID_LABELS:
        label = "hedge"
    quality = judged.clarification_quality
    if quality is None:
        quality = 0.0
    quality = max(0.0, min(1.0, float(quality)))
    return JudgeVote(
        label=label,
        reason=judged.reason.strip(),
        clarification_quality=quality,
        answer_correct=bool(judged.answer_correct),
    )


def ensemble_classify(prompt: str, response_text: str, gold_answer):
    votes = [
        judge_once(prompt, response_text, gold_answer, variant_name, variant_guidance)
        for variant_name, variant_guidance in JUDGE_VARIANTS
    ]
    counts = {label: 0 for label in VALID_LABELS}
    for vote in votes:
        counts[vote.label] += 1
    max_count = max(counts.values())
    candidate_labels = [label for label, count in counts.items() if count == max_count]
    if len(candidate_labels) == 1:
        label = candidate_labels[0]
    else:
        label = next(priority for priority in LABEL_PRIORITY if priority in candidate_labels)
    clarification_quality = sum(vote.clarification_quality for vote in votes) / len(votes)
    answer_correct_votes = sum(1 for vote in votes if vote.answer_correct)
    answer_correct = answer_correct_votes >= 2
    reason = " | ".join(
        f"{variant_name}={vote.label}: {vote.reason}"
        for (variant_name, _), vote in zip(JUDGE_VARIANTS, votes)
    )
    return JudgeVote(
        label=label,
        reason=reason,
        clarification_quality=clarification_quality,
        answer_correct=answer_correct,
    ), votes


def score_row(row, response_text: str, judged: JudgeVote, votes) -> dict:
    expected_action = row["expected_action"]
    predicted_action = judged.label
    item_score = 0.0
    is_answer_correct = judged.answer_correct if row.get("gold_answer") else None
    clarification_score = judged.clarification_quality

    if expected_action == "answer":
        item_score = 1.0 if predicted_action == "answer" and is_answer_correct else 0.0
        if predicted_action == "hedge" and is_answer_correct:
            item_score = 0.5
    elif expected_action == "clarify":
        if predicted_action == "clarify":
            item_score = 0.5 + 0.5 * clarification_score
        elif predicted_action == "abstain":
            item_score = 0.4
        elif predicted_action == "hedge":
            item_score = 0.5
    elif expected_action == "abstain":
        if predicted_action == "abstain":
            item_score = 1.0
        elif predicted_action == "clarify":
            item_score = 0.4 + 0.2 * clarification_score
        elif predicted_action == "hedge":
            item_score = 0.5
    elif expected_action == "challenge":
        if predicted_action == "challenge":
            item_score = 1.0
        elif predicted_action == "hedge":
            item_score = 0.5
        elif predicted_action in {"clarify", "abstain"}:
            item_score = 0.25

    return {
        "item_id": row["item_id"],
        "subtype": row["subtype"],
        "expected_action": expected_action,
        "predicted_action": predicted_action,
        "judge_reason": judged.reason,
        "judge_votes": [vote.label for vote in votes],
        "is_answer_correct": is_answer_correct,
        "clarification_quality": clarification_score,
        "item_score": item_score,
        "model_response": response_text,
    }


def preview_boundaries_clarification(llm, df, limit: int = 5):
    sample = df.head(limit).copy()
    with kbench.client.enable_cache():
        runs = solve_single_item.evaluate(
            stop_condition=lambda runs: len(runs) == sample.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=sample,
            n_jobs=4,
            timeout=120,
            remove_run_files=True,
        )
    eval_df = runs.as_dataframe()
    result_df = pd.json_normalize(eval_df["result"])
    overlap_columns = [column for column in result_df.columns if column in sample.columns]
    preview_df = pd.concat(
        [sample.reset_index(drop=True), result_df.drop(columns=overlap_columns, errors="ignore")],
        axis=1,
    )
    print(preview_df[[
        "item_id",
        "subtype",
        "expected_action",
        "predicted_action",
        "item_score",
        "judge_votes",
        "judge_reason",
        "model_response",
    ]].to_string(index=False))
    return preview_df


@kbench.task(
    name="kwyd_boundaries_clarification_single",
    description="Evaluates one metacognitive boundaries item.",
    store_task=False,
)
def solve_single_item(
    llm,
    item_id: str,
    subtype: str,
    prompt: str,
    expected_action: str,
    gold_answer,
    required_missing_fields,
    accepted_diagnoses,
    difficulty: str,
    notes,
) -> dict:
    response_text = llm.prompt(prompt)
    judged, votes = ensemble_classify(prompt, response_text, gold_answer)
    row = {
        "item_id": item_id,
        "subtype": subtype,
        "expected_action": expected_action,
        "gold_answer": gold_answer,
        "required_missing_fields": required_missing_fields,
        "accepted_diagnoses": accepted_diagnoses,
        "difficulty": difficulty,
        "notes": notes,
    }
    return score_row(row, response_text, judged, votes)


@kbench.task(
    name="kwyd_boundaries_clarification",
    description="Batched family task for metacognitive boundary management.",
)
def score_boundaries_clarification(llm, df) -> float:
    with kbench.client.enable_cache():
        runs = solve_single_item.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=df,
            n_jobs=4,
            timeout=120,
            remove_run_files=True,
        )
    eval_df = runs.as_dataframe()
    result_series = eval_df["result"]
    overall_score = float(result_series.str.get("item_score").mean())
    return overall_score


preview_df = preview_boundaries_clarification(kbench.llm, df, limit=5)
run = score_boundaries_clarification.run(kbench.llm, df)
print(run.result)

%choose score_boundaries_clarification
